In [ ]:
#This code shows a modified version of the Azimuth/Elevation diagrams that takes in a Satellite Name, line1 and line2 of a TLE,
# the year, month, day and time of the beginning of the pass, the length of time of the pass, and the duration of the pass

from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import ListedColormap
def add_arrow(ax, start, end, color='cyan', arrow_size=12):
    """
    Add an arrow from start to end on the given Axes.
    """
    arrow = FancyArrowPatch(
        posA=start, posB=end,
        arrowstyle='-|>',  # arrow head style
        mutation_scale=arrow_size,
        color=color,
        linewidth=1
    )
    ax.add_patch(arrow)
    
def AltitudeAzimuthDiagramArrowhead(ObjectName, line1, line2, location, year, month, day, hour, minute, second, ObservationLengthInMinutes, LowestVisibleAltitude):
    import skyfield
    from skyfield.api import load, wgs84, EarthSatellite
    import numpy as np
    from datetime import datetime
    from pytz import timezone
    import matplotlib.pyplot as plt
    from matplotlib.collections import LineCollection
    
    # List of all the locations of the telescopes
    mountpleasant = wgs84.latlon(-42.8055, 147.438)
    Yarragadee = wgs84.latlon(-29.04567, 115.34876)
    Katherine = wgs84.latlon(-14.375464, 132.152373)
    Ceduna = wgs84.latlon(-31.86883, 133.809799)
    newZealand = wgs84.latlon(-45.03861, 170.0958)
    
    ts = load.timescale()
    satellite = EarthSatellite(line1, line2, ObjectName, ts)
    difference = satellite - mountpleasant
    
    #Sets the location
    if location == 'Hb':
        difference = satellite - mountpleasant
    elif location == 'Yg':
        difference = satellite - Yarragadee
    elif location == 'Ke':
        difference = satellite - Katherine
    elif location == 'Cd':
        difference = satellite - Ceduna
    elif location == 'NZ':
        difference = satellite - newZealand
        
        
        #create the observation lists from the TLE
    altitudeList = []
    azimuthList = []
    DistanceList = []
    visibletime = []
    altDifference = 0
    azDifference = 0
    altTime = 1 #second
    azTime = 1 #second
    
#    MaxAltSpeed = 0
#    MaxAzSpeed = 0
    
    LengthofObservation = ObservationLengthInMinutes*60 #(time in minutes)
    for i in range(LengthofObservation):
        #time and location now
        observationtime = ts.utc(year, month, day, hour, minute, second+i)
        topocentricHb = difference.at(observationtime)
        altHb, azHb, heightHb = topocentricHb.altaz()
        individualalt = [altHb.degrees]
        individualaz = [azHb.degrees]
        #time and location 1 second later
        observationtimeplus1 = ts.utc(year, month, day, hour, minute, second+i+1)
        topocentricHb1 = difference.at(observationtimeplus1)
        altHb1, azHb1, heightHb1 = topocentricHb1.altaz()
        individualalt1 = [altHb1.degrees]
        individualaz1 = [azHb1.degrees]
        individialHeight = [heightHb1.km]
        
        if altHb.degrees > LowestVisibleAltitude:
            altitudeList = np.append(altitudeList, individualalt, axis = 0)
            azimuthList = np.append(azimuthList, individualaz, axis = 0)
            DistanceList = np.append(DistanceList, individialHeight, axis = 0)
            visibletime.append(str(observationtime.utc_strftime()))
            
            #Finding the Maximum Altitude Speed
            if altDifference < abs(altHb.degrees-altHb1.degrees):
                altDifference = abs(altHb.degrees-altHb1.degrees)
                MaxAltSpeed = altDifference/altTime
            
            #Finding the Maximum Azimuthal Speed
            if azDifference < abs(azHb.degrees-azHb1.degrees):
                azDifference = abs(azHb.degrees-azHb1.degrees)
                MaxAzSpeed = azDifference/azTime
            
            
            
    AltitudeAzimuth = np.column_stack((altitudeList, azimuthList))

    azimuthRing = [i*np.pi/180 for i in range(361)]
    altitudeRing = 5*np.ones(361)
    
    #Create plot
    plt.style.use('dark_background')#, color = "lime")
    fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})
    plt.style.use('dark_background')
    ax = plt.subplot(polar=True)

    #horizon ring at five degrees
    ax.plot(azimuthRing, altitudeRing, color ="lime", linewidth=1.0)
    ax.set_theta_zero_location("N") #Puts zero at the top
    ax.set_theta_direction(-1) # Makes the graph go clockwise

    ax.set_rlim(90, -45, 1)
    ax.set_rmax(2)
    ax.set_rticks([10, 20, 30, 40, 50, 60, 70, 80, 90])#  # Radial ticks
    ax.set_rlabel_position(-22.5)  # Move radial labels away from plotted line
    ax.grid(True)

    ax.tick_params(axis='x', colors='lime') #tick colours
    ax.tick_params(axis='y', colors='lime')

    ax.spines['polar'].set_color('lime')
    ax.spines['polar'].set_linewidth(1)

    ax.xaxis.grid(True,color='lime',linestyle='-') #grid colours
    ax.yaxis.grid(True,color='lime',linestyle='-')

    titlestring = 'Object: '+ObjectName+ '\n'+ 'Location: '+location+'\n'+'Rise Time: '+str(visibletime[0])+'\n'+'Set  Time: '+str(visibletime[-1])

    ax.set_title(titlestring, va='bottom', color = "white")

    x    = np.linspace(0,1, 100)
    y    = np.linspace(0,1, 100)
    cols = np.linspace(0,1,len(x))

    points = np.array([azimuthList*np.pi/180,altitudeList]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    #cmap=
    single_color_cmap = ListedColormap(["#00FFFF"])

    lc = LineCollection(segments, cmap=single_color_cmap, linestyles='solid') #YlGn colors='cyan',
    lc.set_array(cols)
    lc.set_linewidth(3)
    line = ax.add_collection(lc)
    print(len(segments))
    #for seg, color in zip(segments, colors):
    #    start, end = seg
    #    add_arrow(ax, start, end, color=color, arrow_size=12)
    Spacing = 25
    for seg in range(int(len(segments)/Spacing)):
        start, end = segments[Spacing*seg]
        #print(ax, start,end,seg)
        add_arrow(ax, start, end, color="cyan", arrow_size=25)
    plt.show()
    
    largestAlt = 0
    largestAz = 0
    
    #ZenithTime = 0
    #ZenithHeight = 0

    #finds the point that the altitude is highest
    for i in range(len(altitudeList)-1):
        if altitudeList[i+1]>altitudeList[i]:
            largestAlt=altitudeList[i+1]
            largestAz=azimuthList[i+1]
            ZenithTime = visibletime[i+1]
            ZenithHeight = DistanceList[i+1]
        finalAlt = altitudeList[i]
        finalAz = azimuthList[i]
        finalHeight = DistanceList[i]
        #ZenithTime = visibletime[i]

    print('Satellite view begins: Altitude = ', altitudeList[0], 'degrees, Azimuth = ', azimuthList[0], 'degrees')
    print('Satellite zenith when: Altitude = ', largestAlt, 'degrees, Azimuth = ', largestAz, 'degrees')
    print('Satellite view ending: Altitude = ', finalAlt, 'degrees, Azimuth = ', finalAz, 'degrees')
    print('Satellite Rising Distance =',DistanceList[0],' km Satellite Zenith Distance =',ZenithHeight,' km Satellite Setting Distance =',finalHeight, 'km')
    print('Maximum Altitudinal Speed:', MaxAltSpeed, 'degrees per second')
    print('Maximum Azimuthal Speed:', MaxAzSpeed, 'degrees per second')
    print("Zenith Time:",ZenithTime)
    plt.style.use('default')